# RoseTTAFold2 для строки 29

Откройте ноутбук в Google Colab, выберите GPU T4 и среду 2026.01 (Python 3.12), затем выполните все ячейки сверху вниз.
Зависимости и модель работают в отдельном окружении, поэтому перезапуск ядра между установкой и расчётом не требуется.
DGL 2.4.0 собирается из официального исходного кода; установка и сборка занимают заметно больше времени, чем само предсказание.
Модель RF2_apr23, seed 0, шесть рециклов, одна цепь, одна модель. MSA получается через MMseqs2. Веса и алгоритм не изменены.

Основа: https://github.com/sokrypton/ColabFold/blob/eb061aa6706959fdffe15e86d27ca08083d6ed8d/RoseTTAFold2.ipynb
Исходный код модели: https://github.com/uw-ipd/RoseTTAFold2/tree/4b273b95ba05ce7524d225bfe1711b8fa76a6a11


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, math, hashlib, shutil, shlex
assert sys.version_info[:2] == (3, 12), 'Выберите среду Colab 2026.01 с Python 3.12'
assert Path('/content').is_dir(), 'Этот ноутбук предназначен для Google Colab'
subprocess.run(['nvidia-smi'], check=True)
sequence = 'MASVQQGEKQLFEKFWRGTFKAVATPRPESIIVASITARKPLPRTEPQNNPVVPAQDGPSEKLGQHLATEPLGTNSWERDKTCRELGATRGHSASHDKDL'
def run_live(command, **kwargs):
    print(command, flush=True)
    process = subprocess.Popen(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Command failed: ' + command)
work = Path('/content/row29-rf2')
work.mkdir(exist_ok=True)
run_live(f'{sys.executable} -m pip install virtualenv')
run_live(f'{sys.executable} -m virtualenv /content/row29-rf2-env')
rf_env = os.environ.copy()
rf_env.pop('PYTHONPATH', None)
rf_env['PATH'] = '/content/row29-rf2-env/bin:' + rf_env['PATH']
rf_env['MPLBACKEND'] = 'Agg'
rf_env['DGLBACKEND'] = 'pytorch'
run_live('python -m pip install "setuptools<81" wheel numpy==1.26.4 scipy networkx psutil tqdm requests matplotlib py3Dmol', env=rf_env)
run_live('python -m pip install torch==2.4.0 --index-url https://download.pytorch.org/whl/cu124', env=rf_env)
run_live('git clone --branch v2.4.0 --depth 1 --recurse-submodules --shallow-submodules https://github.com/dmlc/dgl.git /content/row29-dgl-source', env=rf_env)
run_live('cmake -S /content/row29-dgl-source -B /content/row29-dgl-source/build -DCMAKE_BUILD_TYPE=Release -DBUILD_TYPE=release -DUSE_CUDA=ON -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda -DCUDA_ARCH_NAME=Turing -DBUILD_GRAPHBOLT=OFF -DBUILD_SPARSE=OFF -DBUILD_TORCH=OFF -DBUILD_CPP_TEST=OFF -DUSE_LIBXSMM=OFF', env=rf_env)
run_live('cmake --build /content/row29-dgl-source/build --parallel 2', env=rf_env)
run_live('python -m pip install --no-deps --no-build-isolation /content/row29-dgl-source/python', env=rf_env)


In [ ]:
gpu_check = "#@title Проверка DGL и вычислений на GPU после перезапуска\nimport sys, torch, numpy as np, dgl\nimport dgl.function as fn\nfrom dgl.nn.functional import edge_softmax\nprint('Python:',sys.version,'Torch:',torch.__version__,'CUDA:',torch.version.cuda,'DGL:',dgl.__version__,'NumPy:',np.__version__)\nassert torch.__version__.split('+')[0] == '2.4.0'\nassert np.__version__ == '1.26.4'\nassert torch.cuda.is_available()\ng=dgl.graph(([0,1,2,0],[1,2,0,2]),device='cuda')\ng.ndata['h']=torch.tensor([[1.],[2.],[3.]],device='cuda')\ng.update_all(fn.copy_u('h','m'),fn.sum('m','out'))\nassert torch.allclose(g.ndata['out'],torch.tensor([[3.],[1.],[3.]],device='cuda'))\na=edge_softmax(g,torch.zeros(4,1,device='cuda'))\nassert torch.allclose(a,torch.tensor([[1.],[.5],[1.],[.5]],device='cuda'))\nprint('DGL CUDA graph/message-passing/edge-softmax OK',torch.cuda.get_device_name(0))\n"
run_live("python -c " + shlex.quote(gpu_check), env=rf_env)


In [ ]:
script = '#@title setup **RoseTTAFold2** (~1m)\nparams = "RF2_apr23" # @param ["RF2_apr23","RF2_jan24"]\n\nimport os, time, sys, subprocess\n\ndef run_shell(command):\n  result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n  if result.returncode:\n    print(result.stdout, flush=True)\n    result.check_returncode()\n  print("OK:", command, flush=True)\n  return result.returncode\n\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"\n\nif params == "RF2_jan24" and not os.path.isfile(f"{params}.tgz"):\n  # send param download into background\n  run_shell("apt-get update -qq && apt-get install -y aria2 && aria2c -q -x 16 https://files.ipd.uw.edu/dimaio/RF2_jan24.tgz")\n\nif params == "RF2_apr23" and not os.path.isfile(f"{params}.tgz"):\n  # send param download into background\n  run_shell("apt-get update -qq && apt-get install -y aria2 && aria2c -q -x 16 https://files.ipd.uw.edu/dimaio/RF2_apr23.tgz")\n\nif not os.path.isdir("RoseTTAFold2"):\n  print("install RoseTTAFold2")\n  run_shell("git clone https://github.com/uw-ipd/RoseTTAFold2.git && git -C RoseTTAFold2 checkout 4b273b95ba05ce7524d225bfe1711b8fa76a6a11")\n  run_shell("pip install py3Dmol")\n\n  # 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies\n  # 25Aug2025: updating dgi install to work with latest pytorch\n  # DGL 2.4.0 is built from official sources in the bootstrap cells.\n  run_shell("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")\n  run_shell("cd RoseTTAFold2/SE3Transformer; pip install .")\n\n  run_shell("wget https://raw.githubusercontent.com/sokrypton/ColabFold/eb061aa6706959fdffe15e86d27ca08083d6ed8d/colabfold/colabfold.py -O colabfold_utils.py")\n\n  #run_shell("pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html")\n  #run_shell("cd RoseTTAFold2/SE3Transformer; pip -q install --no-cache-dir -r requirements.txt; pip -q install .")\n  #run_shell("wget https://raw.githubusercontent.com/sokrypton/ColabFold/beta/colabfold/mmseqs/api.py")\n\n  # install hhsuite\n  print("install hhsuite")\n  os.makedirs("hhsuite", exist_ok=True)\n  run_shell(f"curl -fsSL https://github.com/soedinglab/hh-suite/releases/download/v3.3.0/hhsuite-3.3.0-SSE2-Linux.tar.gz | tar xz -C hhsuite/")\n\nif not os.path.isfile(f"{params}.pt"):\n  time.sleep(5)\n\nif os.path.isfile(f"{params}.tgz.aria2"):\n  print("downloading RoseTTAFold2 params")\n  while os.path.isfile(f"{params}.tgz.aria2"):\n    time.sleep(5)\n\nif not os.path.isfile(f"{params}.pt"):\n  run_shell(f"tar -zxvf {params}.tgz")\n  if params == "RF2_apr23":\n    run_shell(f"mv weights/{params}.pt .")\n\nif not "IMPORTED" in dir():\n  if \'RoseTTAFold2/network\' not in sys.path:\n    os.environ["DGLBACKEND"] = "pytorch"\n    sys.path.append(\'RoseTTAFold2/network\')\n  if "hhsuite" not in os.environ[\'PATH\']:\n    os.environ[\'PATH\'] += ":hhsuite/bin:hhsuite/scripts"\n\n  import matplotlib.pyplot as plt\n  import numpy as np\n  from parsers import parse_a3m\n  #from api import run_mmseqs2\n  from colabfold_utils import run_mmseqs2\n  import py3Dmol\n  import torch\n  from string import ascii_uppercase, ascii_lowercase\n  import hashlib, re, os\n  import random\n\n  def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()\n  alphabet_list = list(ascii_uppercase+ascii_lowercase)\n  from collections import OrderedDict, Counter\n\n  IMPORTED = True\n\nif not "pred" in dir() or params_sele != params:\n  from predict import Predictor\n  print("compile RoseTTAFold2")\n\n  if (torch.cuda.is_available()):\n    pred = Predictor(f"{params}.pt", torch.device("cuda:0"))\n  else:\n    print ("WARNING: using CPU")\n    pred = Predictor(f"{params}.pt", torch.device("cpu"))\n  params_sele = params\n\ndef get_unique_sequences(seq_list):\n    unique_seqs = list(OrderedDict.fromkeys(seq_list))\n    return unique_seqs\n\ndef run_mmseqs2_wrapper(*args, **kwargs):\n  kwargs[\'user_agent\'] = "colabfold/rosettafold2"\n  return run_mmseqs2(*args, **kwargs)\n\ndef get_msa(seq, jobname, cov=50, id=90, max_msa=2048,\n            mode="unpaired_paired"):\n\n  assert mode in ["unpaired","paired","unpaired_paired"]\n  seqs = [seq] if isinstance(seq,str) else seq\n\n  # collapse homooligomeric sequences\n  counts = Counter(seqs)\n  u_seqs = list(counts.keys())\n  u_nums = list(counts.values())\n\n  # expand homooligomeric sequences\n  first_seq = "/".join(sum([[x]*n for x,n in zip(u_seqs,u_nums)],[]))\n  msa = [first_seq]\n\n  path = os.path.join(jobname,"msa")\n  os.makedirs(path, exist_ok=True)\n  if mode in ["paired","unpaired_paired"] and len(u_seqs) > 1:\n    print("getting paired MSA")\n    out_paired = run_mmseqs2_wrapper(u_seqs, f"{path}/", use_pairing=True)\n    headers, sequences = [],[]\n    for a3m_lines in out_paired:\n      n = -1\n      for line in a3m_lines.split("\\n"):\n        if len(line) > 0:\n          if line.startswith(">"):\n            n += 1\n            if len(headers) < (n + 1):\n              headers.append([])\n              sequences.append([])\n            headers[n].append(line)\n          else:\n            sequences[n].append(line)\n    # filter MSA\n    with open(f"{path}/paired_in.a3m","w") as handle:\n      for n,sequence in enumerate(sequences):\n        handle.write(f">n{n}\\n{\'\'.join(sequence)}\\n")\n    run_shell(f"hhfilter -i {path}/paired_in.a3m -id {id} -cov {cov} -o {path}/paired_out.a3m")\n    with open(f"{path}/paired_out.a3m","r") as handle:\n      for line in handle:\n        if line.startswith(">"):\n          n = int(line[2:])\n          xs = sequences[n]\n          # expand homooligomeric sequences\n          xs = [\'/\'.join([x]*num) for x,num in zip(xs,u_nums)]\n          msa.append(\'/\'.join(xs))\n\n  if len(msa) < max_msa and (mode in ["unpaired","unpaired_paired"] or len(u_seqs) == 1):\n    print("getting unpaired MSA")\n    out = run_mmseqs2_wrapper(u_seqs,f"{path}/")\n    Ls = [len(seq) for seq in u_seqs]\n    sub_idx = []\n    sub_msa = []\n    sub_msa_num = 0\n    for n,a3m_lines in enumerate(out):\n      sub_msa.append([])\n      with open(f"{path}/in_{n}.a3m","w") as handle:\n        handle.write(a3m_lines)\n      # filter\n      run_shell(f"hhfilter -i {path}/in_{n}.a3m -id {id} -cov {cov} -o {path}/out_{n}.a3m")\n      with open(f"{path}/out_{n}.a3m","r") as handle:\n        for line in handle:\n          if not line.startswith(">"):\n            xs = [\'-\'*l for l in Ls]\n            xs[n] = line.rstrip()\n            # expand homooligomeric sequences\n            xs = [\'/\'.join([x]*num) for x,num in zip(xs,u_nums)]\n            sub_msa[-1].append(\'/\'.join(xs))\n            sub_msa_num += 1\n      sub_idx.append(list(range(len(sub_msa[-1]))))\n\n    while len(msa) < max_msa and sub_msa_num > 0:\n      for n in range(len(sub_idx)):\n        if len(sub_idx[n]) > 0:\n          msa.append(sub_msa[n][sub_idx[n].pop(0)])\n          sub_msa_num -= 1\n        if len(msa) == max_msa:\n          break\n\n  with open(f"{jobname}/msa.a3m","w") as handle:\n    for n,sequence in enumerate(msa):\n      handle.write(f">n{n}\\n{sequence}\\n")\n#@title ###run **RoseTTAFold2**\nsequence = "MASVQQGEKQLFEKFWRGTFKAVATPRPESIIVASITARKPLPRTEPQNNPVVPAQDGPSEKLGQHLATEPLGTNSWERDKTCRELGATRGHSASHDKDL" #@param {type:"string"}\njobname = "assignment5_row29_rf2" #@param {type:"string"}\n\n#@markdown symmetry settings\nsym = "X" #@param ["X","C", "D", "T", "I", "O"]\norder = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}\nmsa_concat_mode = "diag" #@param ["diag", "repeat", "default"]\n\n#@markdown msa settings\nmsa_method = "mmseqs2" #@param ["mmseqs2","single_sequence","custom_a3m"]\npair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}\ncollapse_identical = False #@param {type:"boolean"}\n\n#@markdown RoseTTAFold2 settings\nnum_recycles = 6 #@param [0, 1, 3, 6, 12, 24] {type:"raw"}\n\n#@markdown stochastic settings\nuse_mlm = False #@param {type:"boolean"}\nuse_dropout = False #@param {type:"boolean"}\nmax_msa = 256 #@param [16, 32, 64, 128, 256, 512] {type:"raw"}\nrandom_seed = 0 #@param {type:"integer"}\nnum_models = 1 #@param ["1", "5", "10", "15", "20", "25"] {type:"raw"}\n\n# process\nmax_extra_msa = max_msa * 8\n\nsequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())\nsequence = re.sub(":+",":",sequence)\nsequence = re.sub("^[:]+","",sequence)\nsequence = re.sub("[:]+$","",sequence)\n\nif sym in ["X","C"]:\n  copies = order\nelif sym in ["D"]:\n  copies = order * 2\nelse:\n  copies = {"T":12,"O":24,"I":60}[sym]\n  order = ""\nsymm = sym + str(order)\n\nsequences = sequence.replace(":","/").split("/")\nif collapse_identical:\n  u_sequences = get_unique_sequences(sequences)\nelse:\n  u_sequences = sequences\nsequences = sum([u_sequences] * copies,[])\nlengths = [len(s) for s in sequences]\n\n# TODO\n#subcrop = 1000 if sum(lengths) > 1400 else -1\n\nsubcrop = -1\ntopk = 1536\n\n\nsequence = "/".join(sequences)\njobname = jobname+"_"+symm+"_"+get_hash(sequence)[:5]\n\nprint(f"jobname: {jobname}")\nprint(f"lengths: {lengths}")\n\nos.makedirs(jobname, exist_ok=True)\nif msa_method == "mmseqs2":\n  get_msa(u_sequences, jobname, mode=pair_mode, max_msa=max_extra_msa)\n\nelif msa_method == "single_sequence":\n  u_sequence = "/".join(u_sequences)\n  with open(f"{jobname}/msa.a3m","w") as a3m:\n    a3m.write(f">{jobname}\\n{u_sequence}\\n")\n\nelif msa_method == "custom_a3m":\n  print("upload custom a3m")\n  msa_dict = files.upload()\n  lines = msa_dict[list(msa_dict.keys())[0]].decode().splitlines()\n  a3m_lines = []\n  for line in lines:\n    line = line.replace("\\x00","")\n    if len(line) > 0 and not line.startswith(\'#\'):\n      a3m_lines.append(line)\n\n  with open(f"{jobname}/msa.a3m","w") as a3m:\n    a3m.write("\\n".join(a3m_lines))\n\nbest_plddt = None\nbest_seed = None\nfor seed in range(random_seed,random_seed+num_models):\n  torch.manual_seed(seed)\n  random.seed(seed)\n  np.random.seed(seed)\n  npz = f"{jobname}/rf2_seed{seed}_00.npz"\n  pred.predict(inputs=[f"{jobname}/msa.a3m"],\n               out_prefix=f"{jobname}/rf2_seed{seed}",\n               symm=symm,\n               ffdb=None, #TODO (templates),\n               n_recycles=num_recycles,\n               msa_mask=0.15 if use_mlm else 0.0,\n               msa_concat_mode=msa_concat_mode,\n               nseqs=max_msa,\n               nseqs_full=max_extra_msa,\n               subcrop=subcrop,\n               topk=topk,\n               is_training=use_dropout)\n  plddt = np.load(npz)["lddt"].mean()\n  if best_plddt is None or plddt > best_plddt:\n    best_plddt = plddt\n    best_seed = seed'
(work/'run_model.py').write_text(script)
run_live('python -u run_model.py', cwd=work, env=rf_env)
run_live('python -m pip freeze > dependencies.txt', cwd=work, env=rf_env)
models = sorted(work.glob('assignment5_row29_rf2*/rf2_seed0_00_pred.pdb'))
assert len(models) == 1, models
amino = dict(zip('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL'.split(), 'ARNDCQEGHILKMFPSTWYV'))
for path in models:
    atoms = [line for line in path.read_text().splitlines() if line.startswith('ATOM')]
    ca = [line for line in atoms if line[12:16].strip() == 'CA']
    assert len(ca) == 100 and {line[21] for line in ca} == {'A'}
    assert ''.join(amino[line[17:20]] for line in ca) == sequence
    assert all(math.isfinite(float(line[a:b])) for line in atoms for a,b in [(30,38),(38,46),(46,54)])
    print(path.name, 'residues', len(ca), 'mean_pLDDT', sum(float(line[60:66]) for line in ca)/100, 'sha256', hashlib.sha256(path.read_bytes()).hexdigest())
print('RoseTTAFold2 prediction validated')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'py3Dmol==2.5.5'], check=True)
import py3Dmol
view = py3Dmol.view(width=750, height=450)
view.addModel(models[0].read_text(), 'pdb')
view.setStyle({'cartoon': {'color': 'gray'}})
view.zoomTo()
view.show()


In [ ]:
from google.colab import files
archive_path = shutil.make_archive('/content/row29_rosettafold2_results', 'zip', work)
files.download(archive_path)
